# DSCI 552 Homework 3
Name: Xiaoyi Kuang  
GitHub username: Xiaoyi-Kuang1  

In [16]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

folder = Path('..')
if not (folder / 'data' / 'AReM').exists():
    folder = Path('.')
data_folder = folder / 'data' / 'AReM'
output_folder = folder / 'outputs'
output_folder.mkdir(exist_ok=True)

activities = ['bending1', 'bending2', 'cycling', 'lying', 'sitting', 'standing', 'walking']
columns = ['time', 'avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']
pd.set_option('display.max_rows', 50)
np.random.seed(552)

## 1(a)

In [17]:
def read_data(file):
    rows = []
    with open(file) as text:
        for line in text:
            line = line.strip()
            if line == '' or line.startswith('#'):
                continue
            values = line.replace(',', ' ').split()
            if len(values) != 7:
                raise ValueError('Unexpected columns in ' + str(file))
            row = []
            for value in values:
                row.append(float(value))
            rows.append(row)
    data = pd.DataFrame(rows, columns=columns)
    return data

files = []
for activity in activities:
    for number in range(1, 16):
        file = data_folder / activity / ('dataset' + str(number) + '.csv')
        if file.exists():
            files.append(file)

check_rows = []
for file in files:
    data = read_data(file)
    row = {'activity': file.parent.name, 'file': file.name, 'rows': len(data), 'columns': data.shape[1], 'missing': int(data.isna().sum().sum())}
    check_rows.append(row)
check = pd.DataFrame(check_rows)
display(check.groupby('activity').agg(files=('file', 'count'), min_rows=('rows', 'min'), max_rows=('rows', 'max')))
display(check[check['rows'] != 480])
check.to_csv(output_folder / 'data_check.csv', index=False)

,files,min_rows,max_rows
activity,,,
bending1,7,480,480
bending2,6,480,480
cycling,15,480,480
lying,15,480,480
sitting,15,479,480
standing,15,480,480
walking,15,480,480


,activity,file,rows,columns,missing
50,sitting,dataset8.csv,479,7,0


## 1(b)

In [18]:
info_rows = []
for file in files:
    activity = file.parent.name
    number = int(file.stem.replace('dataset', ''))
    group = 'train'
    if activity in ['bending1', 'bending2']:
        if number <= 2:
            group = 'test'
    elif number <= 3:
        group = 'test'
    row = {'instance': len(info_rows) + 1, 'activity': activity, 'file': file.name, 'set': group}
    info_rows.append(row)
info = pd.DataFrame(info_rows)
display(pd.crosstab(info['activity'], info['set']))
print(info['set'].value_counts())
info.to_csv(output_folder / 'sample_info.csv', index=False)

set,test,train
activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12


set
train    69
test     19
Name: count, dtype: int64


## 1(c)(i)
Common time-domain features include minimum, maximum, mean, median, standard deviation, variance, quartiles, interquartile range, root mean square, skewness, kurtosis, absolute sum of changes, and autocorrelation.

## 1(c)(ii)

In [19]:
feature_rows = []
for file in files:
    data = read_data(file)
    row = {}
    for number in range(1, 7):
        values = data.iloc[:, number]
        name = str(number)
        row['min' + name] = values.min()
        row['max' + name] = values.max()
        row['mean' + name] = values.mean()
        row['median' + name] = values.median()
        row['std' + name] = values.std(ddof=1)
        row['1st quart' + name] = values.quantile(0.25)
        row['3rd quart' + name] = values.quantile(0.75)
    feature_rows.append(row)
features = pd.DataFrame(feature_rows)
features.index = info['instance']
features.index.name = 'instance'
print('Feature table shape:', features.shape)
display(features)
features.to_csv(output_folder / 'features.csv')

all_data = pd.concat([info.reset_index(drop=True), features.reset_index(drop=True)], axis=1)
train = all_data[all_data['set'] == 'train'].copy()
test = all_data[all_data['set'] == 'test'].copy()
train.to_csv(output_folder / 'train_features.csv', index=False)
test.to_csv(output_folder / 'test_features.csv', index=False)

Feature table shape: (88, 42)


,min1,max1,mean1,median1,std1,1st quart1,3rd quart1,min2,max2,mean2,...,std5,1st quart5,3rd quart5,min6,max6,mean6,median6,std6,1st quart6,3rd quart6
instance,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,0.358604,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,0.372437,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,4.576562,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
85,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,4.456333,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
86,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,4.371958,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350


## 1(c)(iii)
For each feature, I draw 88 values with replacement and calculate the standard deviation. I repeat this 1,000 times. I use the 5th and 95th percentiles as the 90% confidence interval.

In [20]:
result_rows = []
for name in features.columns:
    values = features[name].to_numpy()
    boot_std = []
    for repeat in range(1000):
        sample = np.random.choice(values, size=len(values), replace=True)
        boot_std.append(np.std(sample, ddof=1))
    low = np.percentile(boot_std, 5)
    high = np.percentile(boot_std, 95)
    row = {'feature': name, 'std': np.std(values, ddof=1), 'ci_low': low, 'ci_high': high}
    result_rows.append(row)
result = pd.DataFrame(result_rows)
display(result.round(4))
result.to_csv(output_folder / 'bootstrap_intervals.csv', index=False)

,feature,std,ci_low,ci_high
0,min1,9.5700,8.2877,10.7176
1,max1,4.3944,3.3862,5.2610
2,mean1,5.3357,4.7107,5.8971
3,median1,5.4401,4.7764,6.0446
4,std1,1.7722,1.5689,1.9567
5,1st quart1,6.1536,5.6059,6.6258
6,3rd quart1,5.1389,4.2758,5.8329
7,min2,0.0000,0.0000,0.0000
8,max2,5.0627,4.6260,5.3889
9,mean2,1.5742,1.3878,1.7024


## 1(c)(iv)
I choose mean, standard deviation, and maximum. Mean shows the average level, standard deviation shows the variation, and maximum shows the highest value.

## 2. ISLR 3.7.4

### (a)
The cubic model has lower or equal training RSS because it can give the same fit as the linear model by setting the coefficients of X^2 and X^3 to zero. Adding these terms cannot increase the minimum training RSS.

### (b)
I expect the linear model to have lower test RSS because the true relationship is linear. The extra terms in the cubic model may fit noise and increase variance. However, this is not guaranteed for every test set.

### (c)
The cubic model still has lower or equal training RSS because it includes the linear model as a special case, even when the true relationship is nonlinear.

### (d)
There is not enough information to tell. If the relationship is close to linear, the linear model may have lower test RSS. If the relationship has strong curvature that a cubic model can fit, the cubic model may have lower test RSS.